# Data Cleaning

In [1]:
import pandas as pd
import numpy as np

## Domain Problem Formulation

Our aim was to derive and validate two prediction rules for ciTBI (clinically-important traumatic brain injury) to identify children at very low risk of ciTBI after blunt head trauma for whom CT might be unnecessary (one rule for children younger than 2 years, the other for children 2 years and older).

## Step 1: Review background information

Information below mainly comes from `data/TBI PUD Documentation 10-08-2013.xlsx`.


### Information on data collection

Trained site investigators and other emergency department physicians recorded patient history, injury mechanism, and symptoms and signs on a standardized data form before knowing imaging results (if imaging was done).

**Exclusion rule** 

Followings are excluded from Public Use Dataset: 

1. Children with trivial injury mechanisms defined by ground-level falls or walking or running into stationary objects, and no signs or symptoms of head trauma other than scalp abrasions and lacerations.
   
2. Patients had penetrating trauma, known brain tumors, pre-existing neurological disorders complicating assessment, or neuroimaging at an outside hospital before transfer.  

3. Patients had ventricular shunts or bleeding disorders.

We have two outcomes specifically defined:

+ **clinically-important TBI (ciTBI)** : a priori as death from TBI, neurosurgery, intubation for more than 24 hours for TBI, or hospital admission of two or more nights for theTBI in association with TBI on CT. This outcome was defined to exclude brief intubations for imaging or overnight admission for minor CT findings.  Hospitalizations for social reasons were not included.

+ **Traumatic brain injury on CT (TBI on CT)**: intracranial hemorrhage or contusion, cerebral edema, traumatic infarction, diffuse axonal injury, shearing injury, sigmoid sinus thrombosis, midline shift of intracranial contents or signs of brain herniation, diastasis of the skull, pneumocephalus, or skull fracture depressed by at least the width of the skull table.


### Data Dictionary

The whole dictionary can be found in `data/TBI PUD Documentation 10-08-2013.xlsx`.

Based on basic understanding, the observation unit should be patient ( identified by Patient Number `PatNum`).

**Remark**:
+ for certain categorical feature, `91`, `92` are used to imply speical cases like not applicable.
+ samples with GCS in 3-13 have been excluded
+ `CTForm1` doesn't indicate whether the patient received the CT, `CTDone` or `EDCT` means whether an actual CT was really ordered
+ 



> **Question: Are all sub-categories consistent with their parent categories?**
>
> For example: `SFxBas` indicates "Signs of basilar skull fracture?", while `SFxBasHem`, `SFxBasOto`, etc. represent specific symptoms of basilar skull fracture. If `SFxBas` is coded as `No`, then all corresponding sub-category values should be `92` (not applicable) to maintain logical consistency.



## Step 2: Loading the data

Load the data and get the first insight

In [2]:
tbi_origin = pd.read_csv('../data/TBI PUD 10-08-2013.csv')
tbi_origin.sample(20)

,PatNum,EmplType,Certification,InjuryMech,High_impact_InjSev,Amnesia_verb,LOCSeparate,LocLen,Seiz,SeizOccur,...,Finding20,Finding21,Finding22,Finding23,DeathTBI,HospHead,HospHeadPosCT,Intub24Head,Neurosurgery,PosIntFinal
29959,29960,5.0,3,8.0,2.0,0.0,0.0,92.0,0.0,92.0,...,92,92,92,92,0.0,0.0,0,0.0,0.0,0.0
2901,2902,3.0,1,12.0,2.0,0.0,0.0,92.0,0.0,92.0,...,92,92,92,92,0.0,0.0,0,0.0,0.0,0.0
37214,37215,4.0,3,1.0,3.0,91.0,0.0,92.0,0.0,92.0,...,0,0,0,0,0.0,0.0,0,0.0,0.0,0.0
25245,25246,5.0,2,8.0,2.0,91.0,0.0,92.0,0.0,92.0,...,92,92,92,92,0.0,0.0,0,0.0,0.0,0.0
22777,22778,3.0,3,2.0,3.0,0.0,0.0,92.0,0.0,92.0,...,92,92,92,92,0.0,0.0,0,0.0,0.0,0.0
20725,20726,5.0,2,12.0,2.0,0.0,0.0,92.0,0.0,92.0,...,92,92,92,92,0.0,0.0,0,0.0,0.0,0.0
42653,42654,5.0,2,11.0,2.0,0.0,0.0,92.0,0.0,92.0,...,92,92,92,92,0.0,0.0,0,0.0,0.0,0.0
29914,29915,5.0,2,8.0,3.0,91.0,0.0,92.0,0.0,92.0,...,92,92,92,92,0.0,0.0,0,0.0,0.0,0.0
26484,26485,5.0,4,10.0,2.0,0.0,0.0,92.0,0.0,92.0,...,92,92,92,92,0.0,0.0,0,0.0,0.0,0.0
29880,29881,3.0,3,4.0,2.0,0.0,0.0,92.0,0.0,92.0,...,92,92,92,92,0.0,0.0,0,0.0,0.0,0.0


In [3]:
# check dataframe shape
tbi_origin.shape

(43399, 125)

This is consistent with Kuppermann's paper and the data documentation:

+ 43399 evaluable patients in total
+ 125 variables



In [4]:
tbi_origin.info()

<class 'pandas.DataFrame'>
RangeIndex: 43399 entries, 0 to 43398
Columns: 125 entries, PatNum to PosIntFinal
dtypes: float64(48), int64(77)
memory usage: 41.4 MB


Due to the data storage strategy, all the variables are displayed as numerical values, although most of them are categorical variables on the contrary. This should be converted for further analysis and modelling.

In [5]:
tbi_origin.isnull().mean().sort_values(ascending=False).head(20)

Dizzy           0.368027
Ethnicity       0.367889
ActNorm         0.076845
Race            0.073919
LocLen          0.058895
Observed        0.054748
Amnesia_verb    0.052904
LOCSeparate     0.043595
Drugs           0.041890
HAStart         0.030692
GCSMotor        0.030185
GCSVerbal       0.029909
GCSEye          0.029678
HASeverity      0.024056
VomitLast       0.022858
CTSed           0.021060
Seiz            0.021014
HemaSize        0.017097
NeuroD          0.015208
HA_verb         0.015023
dtype: float64

The largest missingness is 30% in `Dizzy` column, which is at the relatively moderate level. But, we still need further investigation

In [6]:
# Patient number should be the unique identifier
tbi_origin['PatNum'].nunique()

43399

It turns out the we have 43399 patients, and it's consistent with our number of observations. It clarifies my previous assumption that the data is on the patient level.

## Step 3: Examine the data

### Dtype Converting

After reading the data description, although most of variables are numerical, they are just categorical variable with its encoding. Also the specical value like `92` means `not applicable`. Before moving forward, let's first convert the data types.

In [7]:
tbi = tbi_origin.copy()
tbi.dtypes

PatNum                  int64
EmplType              float64
Certification           int64
InjuryMech            float64
High_impact_InjSev    float64
                       ...   
HospHead              float64
HospHeadPosCT           int64
Intub24Head           float64
Neurosurgery          float64
PosIntFinal           float64
Length: 125, dtype: object

In [8]:
numeric_cols = ['GCSTotal','AgeInMonth','AgeinYears','GCSMotor','GCSVerbal','GCSEye']
categorical_cols = tbi.columns.difference(numeric_cols + ['PatNum','EmplType','Certification']).tolist() # get all columns except numeric columns and identifier columns
binary_cols = []
multi_category_cols = []
for col in categorical_cols:
    unique_values = tbi[col].nunique()
    if unique_values == 2:
        binary_cols.append(col)
    elif unique_values > 2:
        multi_category_cols.append(col)
print(f"Binary columns: {binary_cols}")
print(f"Multi-category columns: {multi_category_cols}")

Binary columns: ['AMS', 'ActNorm', 'AgeTwoPlus', 'CTDone', 'CTForm1', 'Clav', 'DeathTBI', 'Dizzy', 'Drugs', 'Ethnicity', 'FontBulg', 'GCSGroup', 'Gender', 'Hema', 'HospHead', 'HospHeadPosCT', 'Intub24Head', 'Intubated', 'NeuroD', 'Neurosurgery', 'OSI', 'Observed', 'Paralyzed', 'PosIntFinal', 'SFxBas', 'Sedated', 'Seiz', 'Vomit']
Multi-category columns: ['AMSAgitated', 'AMSOth', 'AMSRepeat', 'AMSSleep', 'AMSSlow', 'Amnesia_verb', 'CTSed', 'CTSedAge', 'CTSedAgitate', 'CTSedOth', 'CTSedRqst', 'ClavFace', 'ClavFro', 'ClavNeck', 'ClavOcc', 'ClavPar', 'ClavTem', 'EDCT', 'EDDisposition', 'Finding1', 'Finding10', 'Finding11', 'Finding12', 'Finding13', 'Finding14', 'Finding2', 'Finding20', 'Finding21', 'Finding22', 'Finding23', 'Finding3', 'Finding4', 'Finding5', 'Finding6', 'Finding7', 'Finding8', 'Finding9', 'HASeverity', 'HAStart', 'HA_verb', 'HemaLoc', 'HemaSize', 'High_impact_InjSev', 'IndAMS', 'IndAge', 'IndAmnesia', 'IndClinSFx', 'IndHA', 'IndHema', 'IndLOC', 'IndMech', 'IndNeuroD', 'Ind

In [9]:
len(binary_cols), len(multi_category_cols)

(28, 88)

In [10]:
# assign appropriate data types
# `PatNum` should be categorical although it is represented as numeric
# set multicategory columns to category dtype, and numeric columns to numeric dtype (coercing errors to NaN)
tbi[multi_category_cols] = tbi[multi_category_cols].astype('category')
tbi[numeric_cols] = tbi[numeric_cols].apply(pd.to_numeric, errors='coerce')
# set binary columns to numeric (0/1) if they are not already, coercing errors to NaN
tbi[binary_cols] = tbi[binary_cols].apply(pd.to_numeric, errors='coerce')

### Sanity check with paper's data description

In the paper, the author has included a detailed description about how many patients really have ciTB in each age group. We can get a quick check, and then dive into our cleaning.

In the paper, we have 43399 evaluable patients, and after excluding patients with GCS 3-13 or missing primary outcome, we have 42412 patients with GCS 14-15. 

Combining the derivation(33785) and validation(8627) group together, we have 10718 (8502+2216) age < 2 years old and 31694(25283+6411) age >= 2 years old. Within each age group, we have 98 (73+25) ciTBI, 278 (215+63) ciTBI respectively.

Just replicate the filtering to confirm the data description.

In [11]:
tbi.drop_duplicates(subset='PatNum').shape

(43399, 125)

Based on the unique identifier `PatNum`, we have no duplicated rows in the orginial data.

In [12]:
tbi['GCSGroup'].value_counts()

GCSGroup
2    42430
1      969
Name: count, dtype: int64

GCSGroup  = 1 means GCS 3-13, the number is exactly the same as what the paper reads.

In [13]:
# explore missingness in `PosIntFinal`  and `GCSGroup` columns
tbi_filtered = tbi.query('GCSGroup == 2 and not PosIntFinal.isnull()')
tbi_filtered.shape

(42412, 125)

after exclusion, the number is still consistent

In [14]:
# Add a summary at the AgeTwoPlus level
summary = tbi_filtered.groupby('AgeTwoPlus').agg({'PatNum': 'count'}).rename(columns={'PatNum': 'Total'})
result = tbi_filtered.groupby(['AgeTwoPlus','PosIntFinal']).agg({'PatNum':'count'}).rename(columns={'PatNum': 'Count'})
result = result.reset_index().merge(summary, on='AgeTwoPlus')
result['Percent'] = result['Count'] / result['Total'] * 100
result

,AgeTwoPlus,PosIntFinal,Count,Total,Percent
0,1,0.0,10620,10718,99.085650
1,1,1.0,98,10718,0.914350
2,2,0.0,31416,31694,99.122862
3,2,1.0,278,31694,0.877138


The results are exactly the same as paper, which means we can replicate the clinical decision rule using the same data.

As for whether to do the same filtering as Kuppermann, I choose to make it a **judgement call**. So, when I want to apply other prediction models, I have the chance to use more data (patients with GCS 3-13).

### Schema Check

Before any fancy logic, I want to verify that key variables obey expected ranges/types.

Since there are 125 variables in total, going over all of them seems too time-consuming, I will go over the variables whose value is not simply yes or no.

In [15]:
# numeric summary statistics
tbi[numeric_cols].describe()

,GCSTotal,AgeInMonth,AgeinYears,GCSMotor,GCSVerbal,GCSEye
count,43399.000000,43399.000000,43399.000000,42089.000000,42101.000000,42111.000000
mean,14.840711,84.429342,6.581419,5.958398,4.925109,3.959369
std,1.032624,66.503854,5.533878,0.376692,0.417289,0.297200
min,3.000000,0.000000,0.000000,1.000000,1.000000,1.000000
25%,15.000000,23.000000,1.000000,6.000000,5.000000,4.000000
50%,15.000000,67.000000,5.000000,6.000000,5.000000,4.000000
75%,15.000000,144.000000,12.000000,6.000000,5.000000,4.000000
max,15.000000,215.000000,17.000000,6.000000,5.000000,4.000000


In [16]:
# wide-range categorical variables
wide_range_cols = []
for col in categorical_cols:
    unique_values = tbi[col].nunique()
    if unique_values > 3:
        wide_range_cols.append(col)
        print(f"Column: {col}, Unique Values: {unique_values}")
        print(tbi[col].value_counts().head(20))
        print("\n")
print(f"Total wide-range categorical columns: {len(wide_range_cols)}")

Column: EDDisposition, Unique Values: 9
EDDisposition
1.0     38539
3.0      2512
5.0       980
4.0       798
90.0      248
2.0       185
6.0        75
8.0        19
7.0        18
Name: count, dtype: int64


Column: HASeverity, Unique Values: 4
HASeverity
92.0    30593
2.0      5671
1.0      5266
3.0       825
Name: count, dtype: int64


Column: HAStart, Unique Values: 5
HAStart
92.0    30593
2.0     10584
3.0       629
4.0       173
1.0        88
Name: count, dtype: int64


Column: HemaLoc, Unique Values: 4
HemaLoc
92.0    26261
1.0      8900
3.0      4422
2.0      3605
Name: count, dtype: int64


Column: HemaSize, Unique Values: 4
HemaSize
92.0    26261
2.0      9597
1.0      3520
3.0      3279
Name: count, dtype: int64


Column: InjuryMech, Unique Values: 13
InjuryMech
8.0     11883
6.0      4733
1.0      3910
90.0     3465
12.0     3158
11.0     3016
10.0     2979
9.0      2908
7.0      2455
4.0      1701
2.0      1433
5.0       901
3.0       556
Name: count, dtype: int64


Column:

After investigation, all the numerical values are in the reasonable ranges and all the categorical variables are consistent with how they are encoded in the description.

In [17]:
binary_cols = []
multi_category_cols = []
for col in categorical_cols:
    unique_values = tbi[col].nunique()
    if unique_values == 2:
        binary_cols.append(col)
    elif unique_values > 2:
        multi_category_cols.append(col)
print(f"Binary columns: {binary_cols}")
print(f"Multi-category columns: {multi_category_cols}")

Binary columns: ['AMS', 'ActNorm', 'AgeTwoPlus', 'CTDone', 'CTForm1', 'Clav', 'DeathTBI', 'Dizzy', 'Drugs', 'Ethnicity', 'FontBulg', 'GCSGroup', 'Gender', 'Hema', 'HospHead', 'HospHeadPosCT', 'Intub24Head', 'Intubated', 'NeuroD', 'Neurosurgery', 'OSI', 'Observed', 'Paralyzed', 'PosIntFinal', 'SFxBas', 'Sedated', 'Seiz', 'Vomit']
Multi-category columns: ['AMSAgitated', 'AMSOth', 'AMSRepeat', 'AMSSleep', 'AMSSlow', 'Amnesia_verb', 'CTSed', 'CTSedAge', 'CTSedAgitate', 'CTSedOth', 'CTSedRqst', 'ClavFace', 'ClavFro', 'ClavNeck', 'ClavOcc', 'ClavPar', 'ClavTem', 'EDCT', 'EDDisposition', 'Finding1', 'Finding10', 'Finding11', 'Finding12', 'Finding13', 'Finding14', 'Finding2', 'Finding20', 'Finding21', 'Finding22', 'Finding23', 'Finding3', 'Finding4', 'Finding5', 'Finding6', 'Finding7', 'Finding8', 'Finding9', 'HASeverity', 'HAStart', 'HA_verb', 'HemaLoc', 'HemaSize', 'High_impact_InjSev', 'IndAMS', 'IndAge', 'IndAmnesia', 'IndClinSFx', 'IndHA', 'IndHema', 'IndLOC', 'IndMech', 'IndNeuroD', 'Ind

In [18]:
# keep a dict for parent-child relationships
parent_child_groups = {
    # Symptoms / history
    "LOCSeparate": ["LocLen"],
    "Seiz": ["SeizOccur", "SeizLen"],
    "HA_verb": ["HASeverity", "HAStart"],
    "Vomit": ["VomitNbr", "VomitStart", "VomitLast"],

    # AMS and sub-symptoms
    "AMS": ["AMSAgitated", "AMSSleep", "AMSSlow", "AMSRepeat", "AMSOth"],

    # Skull fracture details
    "SFxPalp": ["SFxPalpDepress"],
    "SFxBas": ["SFxBasHem", "SFxBasOto", "SFxBasPer", "SFxBasRet", "SFxBasRhi"],

    # Hematoma details
    "Hema": ["HemaLoc", "HemaSize"],

    # Trauma above clavicles by region
    "Clav": ["ClavFace", "ClavNeck", "ClavFro", "ClavOcc", "ClavPar", "ClavTem"],

    # Neuro deficit details
    "NeuroD": ["NeuroDMotor", "NeuroDSensory", "NeuroDCranial", "NeuroDReflex", "NeuroDOth"],

    # Other substantial injury details
    "OSI": ["OSIExtremity", "OSICut", "OSICspine", "OSIFlank", "OSIAbdomen", "OSIPelvis", "OSIOth"],

    # CT ordered/obtained: indication checklist + sedation
    "CTForm1": [
        "IndAge", "IndAmnesia", "IndAMS", "IndClinSFx", "IndHA", "IndHema", "IndLOC",
        "IndMech", "IndNeuroD", "IndRqstMD", "IndRqstParent", "IndRqstTrauma",
        "IndSeiz", "IndVomit", "IndXraySFx", "IndOth",
        "CTSed"
    ],

    # Sedation reasons
    "CTSed": ["CTSedAgitate", "CTSedAge", "CTSedRqst", "CTSedOth"],

    # CT performed: CT-location + PI findings
    "CTDone": [
        "EDCT", "PosCT",
        "Finding1", "Finding2", "Finding3", "Finding4", "Finding5", "Finding6", "Finding7", "Finding8",
        "Finding9", "Finding10", "Finding11", "Finding12", "Finding13", "Finding14",
        "Finding20", "Finding21", "Finding22", "Finding23"
    ],
}

### Missing Values

In the previous checking, I find that there do exist missing values. As for different variable, we may choose different strategies.

In [19]:
# since our lables are `PosIntFinal`, the missingness in it may affect our analysis
# we will drop rows with missing `PosIntFinal` for our analysis
tbi_nonmiss = tbi.dropna(subset=['PosIntFinal']).copy()
tbi_nonmiss.shape

(43379, 125)

In [20]:
# show all the columns with missing values
tbi_nonmiss.isnull().mean()[tbi_nonmiss.isnull().mean() > 0].shape

(47,)

In [21]:
missing_cols = tbi_nonmiss.columns[tbi_nonmiss.isnull().sum()> 0].to_list()
print(missing_cols)

['EmplType', 'InjuryMech', 'High_impact_InjSev', 'Amnesia_verb', 'LOCSeparate', 'LocLen', 'Seiz', 'SeizOccur', 'SeizLen', 'ActNorm', 'HA_verb', 'HASeverity', 'HAStart', 'Vomit', 'VomitNbr', 'VomitStart', 'VomitLast', 'Dizzy', 'Intubated', 'Paralyzed', 'Sedated', 'GCSEye', 'GCSVerbal', 'GCSMotor', 'AMS', 'SFxPalp', 'SFxPalpDepress', 'FontBulg', 'SFxBas', 'Hema', 'HemaLoc', 'HemaSize', 'Clav', 'NeuroD', 'OSI', 'Drugs', 'CTForm1', 'CTSed', 'Gender', 'Ethnicity', 'Race', 'Observed', 'EDDisposition', 'DeathTBI', 'HospHead', 'Intub24Head', 'Neurosurgery']


In [22]:
set(missing_cols).intersection(numeric_cols)

{'GCSEye', 'GCSMotor', 'GCSVerbal'}

It seems that except GCS compoents(3), all the variables with missing values are categorical(44). 
I will explore them separately.

we have 47 columns which contain missing values, I will focus on dealing with columns with relatively high missing ratio

In [23]:
tbi_nonmiss.isnull().mean().sort_values(ascending=False).head(47)

Dizzy                 0.367989
Ethnicity             0.367828
ActNorm               0.076765
Race                  0.073930
LocLen                0.058876
Observed              0.054773
Amnesia_verb          0.052906
LOCSeparate           0.043616
Drugs                 0.041887
HAStart               0.030706
GCSMotor              0.030176
GCSVerbal             0.029899
GCSEye                0.029669
HASeverity            0.024044
VomitLast             0.022845
CTSed                 0.021070
Seiz                  0.021024
HemaSize              0.017082
NeuroD                0.015192
HA_verb               0.015007
SFxBas                0.010235
Vomit                 0.010189
VomitStart            0.009521
Sedated               0.007815
Paralyzed             0.007746
High_impact_InjSev    0.007700
Intubated             0.007492
AMS                   0.007285
Hema                  0.007031
InjuryMech            0.006939
VomitNbr              0.006847
HemaLoc               0.004864
OSI     

In [24]:
tbi_nonmiss[numeric_cols].isnull().mean()

GCSTotal      0.000000
AgeInMonth    0.000000
AgeinYears    0.000000
GCSMotor      0.030176
GCSVerbal     0.029899
GCSEye        0.029669
dtype: float64

It turns out the missing value mainly appears in the categorical columns

After going through all the columns containing missing values, I will categorize them into several groups for inspection:

+ GCSs:  `GCSEye`, `GCSMotor`, `GCSVerbal`

+ Vomit: `Vomit`, `VomitStart`, `VomitNbr`, `VomitLast`

+ Seizure: `Seiz`, `SeizLen`, `SeizOccur`

+ Palpable skull fracture: `SFxPalp`, `SFxPalpDepress`

+ Remaining columns without any relationship with others: `Dizzy`, ...

#### GCS groups

`GCSMotor`, `GCSVerbal`,`GCSEye` all have missing values. It is directly related to `GCSTotal` which we attach high attention to. 

In [25]:
tbi_nonmiss['GCSTotal'].isnull().sum()

np.int64(0)

It turns out that all the GCS scores are non-missing

In [26]:
tbi_nonmiss.query('GCSMotor.isnull() or GCSVerbal.isnull() or GCSEye.isnull()')\
    [['GCSMotor','GCSTotal','GCSVerbal', 'GCSEye']].sample(20)

,GCSMotor,GCSTotal,GCSVerbal,GCSEye
26820,NaN,15,NaN,NaN
39492,NaN,15,NaN,NaN
27647,NaN,15,NaN,NaN
8426,NaN,15,NaN,NaN
4884,NaN,15,NaN,NaN
38615,NaN,14,NaN,NaN
10384,NaN,15,NaN,NaN
14962,NaN,15,NaN,NaN
19325,NaN,15,NaN,NaN
6373,NaN,15,NaN,NaN


In [27]:
tbi_nonmiss.query('GCSMotor.isnull() or GCSVerbal.isnull() or GCSEye.isnull()')\
    [['GCSMotor','GCSTotal','GCSVerbal', 'GCSEye']].shape

(1341, 4)

In [28]:
# show summary statistics for GCS components
tbi_nonmiss[['GCSMotor','GCSTotal','GCSVerbal', 'GCSEye']].astype('float').describe()

,GCSMotor,GCSTotal,GCSVerbal,GCSEye
count,42070.000000,43379.000000,42082.000000,42092.000000
mean,5.958426,14.840891,4.925265,3.959375
std,0.376655,1.032416,0.416844,0.297229
min,1.000000,3.000000,1.000000,1.000000
25%,6.000000,15.000000,5.000000,4.000000
50%,6.000000,15.000000,5.000000,4.000000
75%,6.000000,15.000000,5.000000,4.000000
max,6.000000,15.000000,5.000000,4.000000


In [29]:
mask_ambiguous_gcs = (
    tbi_nonmiss["GCSTotal"].notna() &                       # total is known
    (tbi_nonmiss[["GCSMotor","GCSVerbal","GCSEye"]]
        .notna()
        .sum(axis=1) == 1)                                 # 0 or 1 component known
)

ambiguous_gcs = tbi_nonmiss.loc[mask_ambiguous_gcs]
print(f"Number of cases with ambiguous GCS: {ambiguous_gcs.shape[0]}")
ambiguous_gcs[['GCSMotor','GCSTotal','GCSVerbal', 'GCSEye']]

Number of cases with ambiguous GCS: 12


,GCSMotor,GCSTotal,GCSVerbal,GCSEye
10175,NaN,13,1.0,NaN
12090,4.0,15,NaN,NaN
15948,NaN,14,4.0,NaN
16740,NaN,15,NaN,4.0
18932,NaN,15,NaN,4.0
19641,NaN,15,NaN,4.0
22122,NaN,13,NaN,2.0
26604,NaN,13,NaN,3.0
27410,NaN,15,NaN,4.0
38729,NaN,15,4.0,NaN


There are only 12 cases which I cannot infer the GCS components

Recall that **GCS = GCSMotor + GCSverbal + GCSEye**

**Inconsistency:**


There exist several types of inconsistency:

+ all the components of GCS are missing, while GCS is 15, we can infer that M = 6, V=5, E = 4, which is the maximum value of each component
  
+ Total GCS is known, and we know two components
  
+ Total GCS is known, but we only know 1 component

The first two scenarios, we can fill the missing value from inferring, but as for the last scenario, we may have no idea. 

For this situation, we may choose replace with mode or just leave the NaN.

**!!!Remark**: If we choose mode-filled or other filling strategy, it bring out the **inconsistency** between total GCS scores and its components.

In [30]:
def impute_gcs(
    df: pd.DataFrame,
    motor_col: str = "GCSMotor",
    verbal_col: str = "GCSVerbal",
    eye_col: str = "GCSEye",
    total_col: str = "GCSTotal"
) -> pd.DataFrame:
    """
    Impute missing GCS components using GCSTotal = Motor + Verbal + Eye.

    Rules:
    - If total==15 and all components missing -> set (6,5,4).
    - If total known and exactly one component missing -> infer it if valid.
    - Invalid inferred values are left as NaN.

    Returns a copy of df with imputed columns.
    """
    out = df.copy()

    # Ensure numeric (preserve NaN)
    for c in [motor_col, verbal_col, eye_col, total_col]:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")

    # Valid ranges
    bounds = {
        motor_col: (1, 6),
        verbal_col: (1, 5),
        eye_col: (1, 4),
        total_col: (3, 15),
    }

    def in_range(val, col):
        lo, hi = bounds[col]
        return pd.notna(val) and (lo <= val <= hi)

    # Row-wise imputation
    def fill_row(row):
        m, v, e, t = row[motor_col], row[verbal_col], row[eye_col], row[total_col]

        m_na, v_na, e_na, t_na = pd.isna(m), pd.isna(v), pd.isna(e), pd.isna(t)
        n_known_comp = sum([not m_na, not v_na, not e_na])

        # Case A: all components missing but total == 15 -> assume max components
        if (n_known_comp == 0) and (not t_na) and (t == 15):
            row[motor_col], row[verbal_col], row[eye_col] = 6, 5, 4
            return row

        # Case B: total known, exactly one component missing -> infer
        if (not t_na) and (n_known_comp == 2):
            if m_na:
                inferred = t - v - e
                row[motor_col] = inferred if in_range(inferred, motor_col) else np.nan
            elif v_na:
                inferred = t - m - e
                row[verbal_col] = inferred if in_range(inferred, verbal_col) else np.nan
            elif e_na:
                inferred = t - m - v
                row[eye_col] = inferred if in_range(inferred, eye_col) else np.nan
            return row
        return row

    out = out.apply(fill_row, axis=1)

    return out

In [31]:
# write a general filling pipline to avoid data leakage in trian-test split
from typing import Dict, List, Literal

FillingStrategy = Literal["mode", "median", "mean"]  

def fit_imputer(train: pd.DataFrame, 
                cols: List[str],
                filling_strategy: FillingStrategy = "mode") -> Dict[str, object]:
    """Return a dict of {col: mode_value} learned from TRAIN only."""
    filling_values = {}
    for c in cols:
        if filling_strategy == "mode":
            s = train[c]
            m = s.mode(dropna=True)
            filling_values[c] = m.iloc[0] if len(m) else np.nan
        elif filling_strategy == "median":
            s = train[c]
            # round median to nearest integer since we only have integer values in our dataset
            filling_values[c] = int(round(s.median()))
        elif filling_strategy == "mean":
            s = train[c]
            # round mean to nearest integer since we only have integer values in our dataset
            filling_values[c] = int(round(s.mean()))
        else:
            raise ValueError("filling_strategy must be mode/median/mean")
    return filling_values

def apply_imputer(df: pd.DataFrame, 
                  filling_values: Dict[str, object]) -> pd.DataFrame:
    """Fill NA using precomputed filling values."""
    out = df.copy()
    for c, v in filling_values.items():
        if c in out.columns and pd.notna(v):
            out[c] = out[c].fillna(v)

    return out

In [32]:
# Usage:
# mode_cols = ["VomitNbr", "VomitStart", "VomitLast"]  # example

# modes = fit_imputer(train, mode_cols)

# train2 = apply_imputer(train, modes)
# val2   = apply_imputer(val, modes)
# test2  = apply_imputer(test, modes)

In [33]:
imputed_gcs_tbi = impute_gcs(tbi_nonmiss)

In [34]:
# compare missingness before and after imputation
before_impute = tbi_nonmiss[['GCSMotor','GCSTotal','GCSVerbal', 'GCSEye']].isnull().sum()
after_impute = imputed_gcs_tbi[['GCSMotor','GCSTotal','GCSVerbal', 'GCSEye']].isnull().sum()
pd.DataFrame({'Before Imputation': before_impute, 'After Imputation': after_impute})

,Before Imputation,After Imputation
GCSMotor,1309,59
GCSTotal,0,0
GCSVerbal,1297,59
GCSEye,1287,54


In [35]:
imputed_gcs_tbi.query('GCSVerbal.isnull() or GCSMotor.isnull() or GCSEye.isnull()')[['GCSTotal','GCSMotor','GCSVerbal', 'GCSEye']]

,GCSTotal,GCSMotor,GCSVerbal,GCSEye
566,15.0,6.0,4.0,NaN
757,13.0,NaN,NaN,NaN
837,14.0,NaN,NaN,NaN
1389,14.0,NaN,NaN,NaN
2281,14.0,NaN,NaN,NaN
...,...,...,...,...
39855,15.0,6.0,NaN,3.0
41812,14.0,NaN,NaN,NaN
42334,3.0,NaN,NaN,NaN
42438,15.0,NaN,NaN,4.0


In [36]:
# at this stage, compute the mode for each component and fill missing values with it for the whole dataset (for simplicity)
mode_cols = ["GCSMotor", "GCSVerbal", "GCSEye"]  # example
modes_gcs = fit_imputer(imputed_gcs_tbi, mode_cols)
tot_imputed_gcs_tbi = apply_imputer(imputed_gcs_tbi, 
                                    modes_gcs
                                    )

In [37]:
tot_imputed_gcs_tbi.loc[imputed_gcs_tbi.query('GCSVerbal.isnull() or GCSMotor.isnull() or GCSEye.isnull()').index, ['GCSTotal','GCSMotor','GCSVerbal', 'GCSEye']]

,GCSTotal,GCSMotor,GCSVerbal,GCSEye
566,15.0,6.0,4.0,4.0
757,13.0,6.0,5.0,4.0
837,14.0,6.0,5.0,4.0
1389,14.0,6.0,5.0,4.0
2281,14.0,6.0,5.0,4.0
...,...,...,...,...
39855,15.0,6.0,5.0,3.0
41812,14.0,6.0,5.0,4.0
42334,3.0,6.0,5.0,4.0
42438,15.0,6.0,5.0,4.0


Actually, after we have inferred the components based on the definition and valid range of GCS score. We have 64 observations in total which we cannot decide the missing values. We can choose to leave the missing value, or just fill with mode.

One **drawback** for the filling is that it may generate invalid scores. For example, we have 11 in total but its components are 6,5,4.

So, I will leave it as a judgment call,

> **Judgment Call**: 
>   Whether to fill the missing valus in GCS component with Mode

#### Vomit groups

In [38]:
tbi_nonmiss[['Vomit','VomitNbr','VomitStart', 'VomitLast']].isnull().sum()

Vomit         442
VomitNbr      297
VomitStart    413
VomitLast     991
dtype: int64

In [39]:
tbi_nonmiss.query('Vomit.isnull() or VomitNbr.isnull() or VomitStart.isnull() or VomitLast.isnull() ')\
    [['Vomit','VomitNbr','VomitStart', 'VomitLast']].sample(10)

,Vomit,VomitNbr,VomitStart,VomitLast
29122,1.0,2.0,NaN,2.0
18243,1.0,3.0,NaN,NaN
38545,1.0,NaN,3.0,1.0
39073,1.0,2.0,NaN,1.0
38457,NaN,92.0,92.0,92.0
38052,1.0,2.0,2.0,NaN
36520,1.0,1.0,3.0,NaN
7335,NaN,92.0,92.0,92.0
32336,1.0,2.0,2.0,NaN
3122,1.0,NaN,NaN,1.0


In [40]:
# whem Vomit == 1, VomitNbr, VomitStart, VomitLast should not be missing. Let's check those cases.
tbi_nonmiss.query('Vomit == 1 and ( VomitNbr.isnull() or VomitStart.isnull() or VomitLast.isnull() )')\
    [['Vomit','VomitNbr','VomitStart', 'VomitLast']]

,Vomit,VomitNbr,VomitStart,VomitLast
24,1.0,1.0,3.0,NaN
59,1.0,1.0,2.0,NaN
64,1.0,2.0,2.0,NaN
76,1.0,NaN,2.0,1.0
161,1.0,NaN,4.0,NaN
...,...,...,...,...
43197,1.0,1.0,2.0,NaN
43214,1.0,2.0,3.0,NaN
43221,1.0,2.0,3.0,NaN
43224,1.0,1.0,2.0,NaN


In [41]:
# check inconsistencies: vomit ==0 but other vomit details present
tbi_nonmiss.query('Vomit == 0 and (not VomitNbr ==92 or not VomitStart == 92 or not VomitLast == 92)').shape

(0, 125)

In [42]:
# check inconsistencies: vomit ==1 but vomit details are missing
tbi_nonmiss.query('Vomit == 1 and (VomitNbr ==92 or  VomitStart == 92 or VomitLast == 92)')[['Vomit','VomitNbr','VomitStart', 'VomitLast']]

,Vomit,VomitNbr,VomitStart,VomitLast


Consistency checking pass! Our logistics are consistent.

As description, if `Vomit` is missing, then other sub-variables should be set to 92

since the missing ratio is quite small, for the missingness, we can make simple judgment calls:

+ if `Vomit` is missing and is binary, add one indicator column `_missing` and for original feature:
    + A: leave it alone
    + B: fill with 0
  
+ if  `Vomit` = 1, and other sub-variables are missing
    + A: add one category named `missing`


In [43]:
# write a general function for parent/child missingness handling
from typing import Dict, List, Literal
ParentMissingFill = Literal["leave", "fill0"]
ChildMissingStrategy = Literal["leave", "missing_category"]
ModeScope = Literal["within_parent_yes", "global"]

def apply_parent_child_missingness(
    df: pd.DataFrame,
    groups: Dict[str, List[str]],
    missing_code: float = 92,
    parent_yes_value: float = 1,
    add_parent_missing_indicator: bool = True,
    parent_missing_strategy: ParentMissingFill = "fill0",  # A: "leave", B: "fill0"
    child_missing_when_parent_yes: ChildMissingStrategy = "missing_category",
    missing_category_label: str = "missing"
) -> pd.DataFrame:
    out = df.copy()

    for parent, children in groups.items():
        # Ensure columns exist
        missing_cols = [c for c in [parent] + children if c not in out.columns]
        if missing_cols:
            raise KeyError(f"Columns not found in df: {missing_cols}")


        # parent missing -> children = missing_code (consistent with definition)
        parent_missing = out[parent].isna()
        out.loc[parent_missing, children] = missing_code

        # add missing indicator for parent missingness
        if add_parent_missing_indicator and parent_missing.any():
            out[f"{parent}_missing"] = parent_missing.astype(int)
        # parent missing fill strategy
        if parent_missing_strategy == "fill0":
            out.loc[parent_missing, parent] = 0
        elif parent_missing_strategy == "leave":
            pass  # do nothing, leave as missing

        # child missing strategy for rows where parent == yes
        parent_yes_mask = out[parent] == parent_yes_value
        if child_missing_when_parent_yes == "leave":
            continue

        if child_missing_when_parent_yes == "missing_category":
            for child in children:
                miss_mask = parent_yes_mask & out[child].isna()
                if not miss_mask.any():
                    continue
                # coerce to categorical if not already, since we need to add a category for missing
                if not isinstance(out[child].dtype, pd.CategoricalDtype):
                     out[child] = out[child].astype("category")

                # Add category then fill
                if missing_category_label not in out[child].cat.categories:
                    out[child] = out[child].cat.add_categories([missing_category_label])

                out.loc[miss_mask, child] = missing_category_label

        else:
            raise ValueError(f"Unknown child_missing_when_parent_yes={child_missing_when_parent_yes}")


    return out

In [44]:
cleaned_vomit = apply_parent_child_missingness(
    tbi_nonmiss,
    groups={"Vomit": ["VomitNbr","VomitStart", "VomitLast"]},
    missing_code=92,
    parent_yes_value=1,
    add_parent_missing_indicator=True,
    parent_missing_strategy="fill0",
    child_missing_when_parent_yes="missing_category",
    missing_category_label="missing"
)
cleaned_vomit.sample(10)

,PatNum,EmplType,Certification,InjuryMech,High_impact_InjSev,Amnesia_verb,LOCSeparate,LocLen,Seiz,SeizOccur,...,Finding21,Finding22,Finding23,DeathTBI,HospHead,HospHeadPosCT,Intub24Head,Neurosurgery,PosIntFinal,Vomit_missing
34121,34122,2.0,1,1.0,2.0,0.0,0.0,92.0,0.0,92.0,...,92,92,92,0.0,0.0,0,0.0,0.0,0.0,0
10200,10201,5.0,3,90.0,2.0,0.0,0.0,92.0,0.0,92.0,...,92,92,92,0.0,0.0,0,0.0,0.0,0.0,0
3427,3428,5.0,2,8.0,2.0,0.0,0.0,92.0,0.0,92.0,...,0,0,0,0.0,0.0,0,0.0,0.0,0.0,0
10491,10492,3.0,3,8.0,3.0,91.0,0.0,92.0,0.0,92.0,...,92,92,92,0.0,0.0,0,0.0,0.0,0.0,0
23576,23577,5.0,2,90.0,2.0,0.0,2.0,1.0,0.0,92.0,...,0,0,0,0.0,0.0,0,0.0,0.0,0.0,0
31619,31620,3.0,3,8.0,3.0,91.0,1.0,2.0,0.0,92.0,...,92,92,92,0.0,0.0,0,0.0,0.0,0.0,0
39563,39564,3.0,3,2.0,3.0,0.0,0.0,92.0,0.0,92.0,...,92,92,92,0.0,0.0,0,0.0,0.0,0.0,0
19652,19653,5.0,2,9.0,2.0,0.0,0.0,92.0,0.0,92.0,...,92,92,92,0.0,0.0,0,0.0,0.0,0.0,0
31259,31260,5.0,3,10.0,2.0,0.0,0.0,92.0,0.0,92.0,...,92,92,92,0.0,0.0,0,0.0,0.0,0.0,0
13890,13891,5.0,3,2.0,3.0,91.0,0.0,92.0,0.0,92.0,...,92,92,92,0.0,0.0,0,0.0,0.0,0.0,0


In [45]:
cleaned_vomit[tbi_nonmiss['Vomit'].isna()][['Vomit','VomitNbr','VomitStart', 'VomitLast','Vomit_missing']].head(10)

,Vomit,VomitNbr,VomitStart,VomitLast,Vomit_missing
2,0.0,92.0,92.0,92.0,1
167,0.0,92.0,92.0,92.0,1
312,0.0,92.0,92.0,92.0,1
351,0.0,92.0,92.0,92.0,1
413,0.0,92.0,92.0,92.0,1
447,0.0,92.0,92.0,92.0,1
499,0.0,92.0,92.0,92.0,1
529,0.0,92.0,92.0,92.0,1
555,0.0,92.0,92.0,92.0,1
669,0.0,92.0,92.0,92.0,1


In [46]:
cleaned_vomit[['Vomit','VomitNbr','VomitStart', 'VomitLast', 'Vomit_missing']].isnull().sum()

Vomit            0
VomitNbr         0
VomitStart       0
VomitLast        0
Vomit_missing    0
dtype: int64

It turns out all the missingness arises from circumstances where `Vomit` = yes, some components are missing.

#### Seizure

columns: `Seiz`, `SeizLen`, `SeizOccur`

In [47]:
tbi_nonmiss[['Seiz','SeizLen','SeizOccur']].isnull().sum()

Seiz         912
SeizLen      115
SeizOccur     70
dtype: int64

In [48]:
tbi_nonmiss.query('Seiz.isnull() or SeizLen.isnull() or SeizOccur.isnull() ')\
    [['Seiz','SeizLen','SeizOccur']].sample(10)

,Seiz,SeizLen,SeizOccur
17490,1.0,NaN,1.0
8612,NaN,92.0,92.0
32875,NaN,92.0,92.0
42620,NaN,92.0,92.0
1030,NaN,92.0,92.0
9731,NaN,92.0,92.0
41103,NaN,92.0,92.0
22188,1.0,NaN,2.0
36548,NaN,92.0,92.0
31943,NaN,92.0,92.0


Quite similar strategy I want to use since the null ratio is quite small:

+ if `Seiz` is `NaN`, set other the sub-cloumns to `92`
+ if `Seiz` is `1`, while other columns are missing, two options: 1. leave `NaN`; 2. fill the missingness with mode

In [49]:
cleaned_seiz = apply_parent_child_missingness(
    tbi_nonmiss,
    groups={"Seiz": ["SeizLen","SeizOccur"]},
    missing_code=92,
    parent_yes_value=1,
    add_parent_missing_indicator=True,
    parent_missing_strategy="fill0",
    child_missing_when_parent_yes="missing_category",
    missing_category_label="missing"
)
cleaned_seiz.sample(10)

,PatNum,EmplType,Certification,InjuryMech,High_impact_InjSev,Amnesia_verb,LOCSeparate,LocLen,Seiz,SeizOccur,...,Finding21,Finding22,Finding23,DeathTBI,HospHead,HospHeadPosCT,Intub24Head,Neurosurgery,PosIntFinal,Seiz_missing
35227,35228,5.0,3,90.0,2.0,0.0,0.0,92.0,0.0,92.0,...,92,92,92,0.0,0.0,0,0.0,0.0,0.0,0
10321,10322,2.0,1,6.0,1.0,0.0,0.0,92.0,0.0,92.0,...,92,92,92,0.0,0.0,0,0.0,0.0,0.0,0
41595,41596,5.0,3,2.0,3.0,0.0,0.0,92.0,0.0,92.0,...,92,92,92,0.0,0.0,0,0.0,0.0,0.0,0
11434,11435,5.0,2,8.0,2.0,0.0,0.0,92.0,0.0,92.0,...,92,92,92,0.0,0.0,0,0.0,0.0,0.0,0
2994,2995,2.0,3,8.0,2.0,91.0,0.0,92.0,0.0,92.0,...,92,92,92,0.0,0.0,0,0.0,0.0,0.0,0
10039,10040,1.0,3,8.0,2.0,0.0,0.0,92.0,0.0,92.0,...,92,92,92,0.0,0.0,0,0.0,0.0,0.0,0
29039,29040,5.0,3,10.0,2.0,1.0,2.0,NaN,0.0,92.0,...,0,0,0,0.0,0.0,0,0.0,0.0,0.0,1
28894,28895,5.0,90,7.0,1.0,91.0,0.0,92.0,0.0,92.0,...,92,92,92,0.0,0.0,0,0.0,0.0,0.0,0
20076,20077,1.0,3,8.0,3.0,91.0,0.0,92.0,0.0,92.0,...,92,92,92,0.0,0.0,0,0.0,0.0,0.0,0
25,26,3.0,3,8.0,2.0,91.0,0.0,92.0,0.0,92.0,...,92,92,92,0.0,0.0,0,0.0,0.0,0.0,0


In [50]:
cleaned_seiz[['Seiz','SeizLen','SeizOccur','Seiz_missing']].isnull().sum()

Seiz            0
SeizLen         0
SeizOccur       0
Seiz_missing    0
dtype: int64

In [51]:
cleaned_seiz[tbi_nonmiss['Seiz'].isna()][['Seiz','SeizLen','SeizOccur','Seiz_missing']].head(10)

,Seiz,SeizLen,SeizOccur,Seiz_missing
2,0.0,92.0,92.0,1
167,0.0,92.0,92.0,1
192,0.0,92.0,92.0,1
215,0.0,92.0,92.0,1
340,0.0,92.0,92.0,1
351,0.0,92.0,92.0,1
397,0.0,92.0,92.0,1
447,0.0,92.0,92.0,1
456,0.0,92.0,92.0,1
482,0.0,92.0,92.0,1


#### Palpable skull fracture

`SFxPalp`, `SFxPalpDepress`

Similar checking and filling strategy will be applied.


In [52]:
tbi_nonmiss[['SFxPalp','SFxPalpDepress']].isnull().sum()

SFxPalp           104
SFxPalpDepress     55
dtype: int64

In [53]:
# take a look at the rows with missing SFxPalp or SFxPalpDepress
tbi_nonmiss.query('SFxPalp.isnull() or SFxPalpDepress.isnull()')[['SFxPalp','SFxPalpDepress']].sample(10)

,SFxPalp,SFxPalpDepress
43326,NaN,92.0
10593,NaN,92.0
33645,NaN,92.0
14782,1.0,NaN
34487,NaN,92.0
38792,NaN,92.0
29057,NaN,92.0
43041,NaN,92.0
1332,NaN,92.0
29725,NaN,92.0


In [54]:
# apply parent/child missingness handling to SFxPalp and SFxPalpDepress
cleaned_sfx = apply_parent_child_missingness(
    tbi_nonmiss,
    groups={"SFxPalp": ["SFxPalpDepress"]},
    missing_code=92,
    parent_yes_value=1,
    add_parent_missing_indicator=True,
    parent_missing_strategy="fill0",
    child_missing_when_parent_yes="missing_category",
    missing_category_label="missing"
)
cleaned_sfx.sample(10)

,PatNum,EmplType,Certification,InjuryMech,High_impact_InjSev,Amnesia_verb,LOCSeparate,LocLen,Seiz,SeizOccur,...,Finding21,Finding22,Finding23,DeathTBI,HospHead,HospHeadPosCT,Intub24Head,Neurosurgery,PosIntFinal,SFxPalp_missing
24155,24156,5.0,3,6.0,1.0,0.0,0.0,92.0,0.0,92.0,...,92,92,92,0.0,0.0,0,0.0,0.0,0.0,0
35907,35908,3.0,1,7.0,1.0,0.0,0.0,92.0,0.0,92.0,...,92,92,92,0.0,0.0,0,0.0,0.0,0.0,0
2851,2852,3.0,3,10.0,2.0,1.0,NaN,92.0,NaN,92.0,...,0,0,0,0.0,0.0,0,0.0,0.0,0.0,0
18437,18438,5.0,2,8.0,2.0,0.0,0.0,92.0,0.0,92.0,...,92,92,92,0.0,0.0,0,0.0,0.0,0.0,0
20987,20988,4.0,2,1.0,3.0,91.0,0.0,92.0,0.0,92.0,...,0,0,0,0.0,0.0,0,0.0,0.0,0.0,0
7304,7305,3.0,3,11.0,2.0,1.0,2.0,NaN,0.0,92.0,...,0,0,0,0.0,0.0,0,0.0,0.0,0.0,0
30261,30262,4.0,3,8.0,3.0,91.0,0.0,92.0,0.0,92.0,...,92,92,92,0.0,0.0,0,0.0,0.0,0.0,0
17614,17615,3.0,3,10.0,2.0,0.0,1.0,3.0,0.0,92.0,...,92,92,92,0.0,0.0,0,0.0,0.0,0.0,0
23773,23774,3.0,3,8.0,3.0,91.0,2.0,2.0,0.0,92.0,...,0,0,0,0.0,0.0,0,0.0,0.0,0.0,0
17485,17486,5.0,1,10.0,2.0,1.0,2.0,NaN,0.0,92.0,...,0,0,0,0.0,0.0,0,0.0,0.0,0.0,0


In [55]:
cleaned_sfx[['SFxPalp','SFxPalpDepress','SFxPalp_missing']].isnull().sum()

SFxPalp            0
SFxPalpDepress     0
SFxPalp_missing    0
dtype: int64

For this missing value handling strategy, we just make sure all the children variables have no missing values either be 92 or the other valid values.

#### Remaining Categorical Variables


As for the other variables which are not associated with other variables, they are all catergorical variables.

Although the most common way to handle this kind of variable is to replace with mode, I don't want to include too many guesses in the cleaning stage. The reasons are as follows: 

1. missing can also be informative. In clinical data, missingness may be systematic (e.g, patient is too unstable, data are not recorded).
2. If I want to fill the missingness with mode, which mode should I use is confusing. Most of the time, it can include my wrong judgements since I am not an expert in it.

I will implement the strategy consistent with the logic when dealing with parent-child missingness.

1. For binary category: add one indicator columns and 
   + A. fill null value with 0; 
   + B. leave it alone
2. For multi-category: add one category called `missing`/ leave it alone


In [56]:
# for the standalone categorical variables with missing values
cate_missing_cols = set(missing_cols).difference(set(['GCSMotor','GCSTotal','GCSVerbal', 'GCSEye',
                                                      'Vomit','VomitNbr','VomitStart', 'VomitLast',
                                                      'Seiz','SeizLen','SeizOccur',
                                                      'SFxPalp','SFxPalpDepress']))

In [57]:
BinaryMissing = Literal["leave", "fill0"]
MultiMissingStrategy = Literal["leave", "missing_category"]

def handle_missing_for_categories(
    df: pd.DataFrame,
    *,
    binary_cat_cols: List[str],
    multi_cat_cols: List[str],
    binary_missing_strategy: BinaryMissing = "fill0",   # A: "leave", B: "fill0"
    add_binary_missing_indicator: bool = True,
    missing_category_label: str = "missing",
    multi_missing_strategy: MultiMissingStrategy = "missing_category",
    force_to_category: bool = True
) -> pd.DataFrame:

    out = df.copy()

    # Validate columns exist 
    all_cols = list(set(binary_cat_cols + multi_cat_cols))
    missing_cols = [c for c in all_cols if c not in out.columns]
    if missing_cols:
        raise KeyError(f"Columns not found in df: {missing_cols}")

    # --- Force dtype to category if requested ---
    if force_to_category:
        # Only convert multi-category columns to category, since binary cats may be 0/1 numeric and that's fine
        for c in multi_cat_cols:
            if not isinstance(out[c].dtype, pd.CategoricalDtype):
                out[c] = out[c].astype("category")

    #  Binary cats: add indicator + fill0/leave 
    for c in binary_cat_cols:
        miss_mask = out[c].isna()

        if add_binary_missing_indicator and miss_mask.any():
            out[f"{c}_missing"] = miss_mask.astype(int)

        if binary_missing_strategy == "fill0":
            out.loc[miss_mask, c] = 0
        elif binary_missing_strategy == "leave":
            pass
        else:
            raise ValueError("binary_missing_strategy must be 'leave' or 'fill0'")

    # Multi-category: add 'missing' level + fill 
    if multi_missing_strategy == "missing_category":
        for c in multi_cat_cols:
            if out[c].isna().any():
                if missing_category_label not in out[c].cat.categories:
                    out[c] = out[c].cat.add_categories([missing_category_label])
                out[c] = out[c].fillna(missing_category_label)
    elif multi_missing_strategy == "leave":
        pass
    else:  
        raise ValueError(f"multi_missing_strategy must be 'missing_category' or 'leave', got {multi_missing_strategy}")

    return out

In [58]:
tbi_standalone_cleaned= handle_missing_for_categories(
    tbi_nonmiss,
    binary_cat_cols= list(set(missing_cols).intersection(set(binary_cols))),
    multi_cat_cols=list(set(missing_cols).intersection(set(multi_category_cols))),
    binary_missing_strategy="fill0",   # or "leave"
)
# check the result

high_impact_miss_marked = tbi_nonmiss['High_impact_InjSev'].isna()
tbi_standalone_cleaned[high_impact_miss_marked][['High_impact_InjSev']].head(10)

,High_impact_InjSev
5,missing
456,missing
477,missing
516,missing
529,missing
707,missing
783,missing
877,missing
1071,missing
1450,missing


### Consistency Checking

#### Parent-child Pairs

There are several parent-children variable pairs. I want to check their consistency:

1. when parent variable is equal to `no` (0), all the children variables should be set to 92.
2. when some of the children variables is `92`, the parent variable should be `NaN` or `no`(0).
   

In [59]:
from typing import Dict, List, Tuple

def check_parent_child_consistency(
    df: pd.DataFrame,
    groups: Dict[str, List[str]],
    missing_code: float = 92,
    parent_no_value: float = 0,
) -> Dict[str, Dict[str, pd.DataFrame]]:
    """
    Check consistency rules for parent/child variable groups.

    Rules
    -----
    1) If parent == parent_no_value, all children must be missing_code.
    2) If any child == missing_code, parent must be NaN or parent_no_value.

    Returns
    -------
    dict
        {parent: {"rule1": df_violations, "rule2": df_violations}}
    """
    results: Dict[str, Dict[str, pd.DataFrame]] = {}

    for parent, children in groups.items():
        missing_cols = [c for c in [parent] + children if c not in df.columns]
        if missing_cols:
            raise KeyError(f"Columns not found in df: {missing_cols}")

        # Rule 1 violations: parent == no but any child != missing_code
        parent_no = df[parent] == parent_no_value
        child_not_missing_code = df[children].ne(missing_code).any(axis=1)
        rule1_mask = parent_no & child_not_missing_code

        # Rule 2 violations: any child == missing_code but parent not NaN and not no
        any_child_missing_code = df[children].eq(missing_code).any(axis=1)
        parent_ok = df[parent].isna() | (df[parent] == parent_no_value) | df[parent].eq(missing_code)
        rule2_mask = any_child_missing_code & (~parent_ok)

        results[parent] = {
            "rule1": df.loc[rule1_mask, [parent] + children],
            "rule2": df.loc[rule2_mask, [parent] + children],
        }

    return results

In [60]:
cols = tbi_nonmiss.columns.tolist()
consistency_results = check_parent_child_consistency(
    tbi_nonmiss,
    groups=parent_child_groups,
    missing_code=92,
    parent_no_value=0,
)

In [61]:
for parent, violations in consistency_results.items():
    rule1_violations = violations["rule1"]
    rule2_violations = violations["rule2"]
    if not rule1_violations.empty:
        print(f"Parent '{parent}' - Rule 1 Violations:")
        print(rule1_violations)
    if not rule2_violations.empty:
        print(f"Parent '{parent}' - Rule 2 Violations:")
        print(rule2_violations)

Parent 'HA_verb' - Rule 2 Violations:
      HA_verb HASeverity HAStart
3        91.0       92.0    92.0
4        91.0       92.0    92.0
5        91.0       92.0    92.0
9        91.0       92.0    92.0
10       91.0       92.0    92.0
...       ...        ...     ...
43383    91.0       92.0    92.0
43385    91.0       92.0    92.0
43389    91.0       92.0    92.0
43391    91.0       92.0    92.0
43392    91.0       92.0    92.0

[14053 rows x 3 columns]
Parent 'SFxPalp' - Rule 2 Violations:
      SFxPalp SFxPalpDepress
35        2.0           92.0
101       2.0           92.0
113       2.0           92.0
192       2.0           92.0
253       2.0           92.0
...       ...            ...
43127     2.0           92.0
43134     2.0           92.0
43190     2.0           92.0
43299     2.0           92.0
43341     2.0           92.0

[976 rows x 2 columns]


There is a special case for `SFxPalp`, which is when palpable skull fracture is unclear exam (2), the children variable `SFxPalpDepress` should also be `92`.

Beyond that, all the pairs are consistent along with the description.

#### Age groups

`AgeInMonth`, `AgeinYears`,`AgeTwoPlus`

Get a deeper look at the age related variables because in our clinical decision rule, `AgeTwoPlus` is a key identifier for grouping.

In [62]:
tbi_nonmiss[['AgeInMonth','AgeinYears','AgeTwoPlus']].describe(include='all')

,AgeInMonth,AgeinYears,AgeTwoPlus
count,43379.000000,43379.000000,43379.000000
mean,84.412711,6.580050,1.748703
std,66.495757,5.533164,0.433764
min,0.000000,0.000000,1.000000
25%,23.000000,1.000000,1.000000
50%,67.000000,5.000000,2.000000
75%,144.000000,12.000000,2.000000
max,215.000000,17.000000,2.000000


In [63]:
# check the maximum age values
tbi_nonmiss.query('AgeInMonth == AgeInMonth.max()')[['AgeInMonth','AgeinYears','AgeTwoPlus']].drop_duplicates()

,AgeInMonth,AgeinYears,AgeTwoPlus
41,215,17,2


In [64]:
# check the minimum age values
tbi_nonmiss.query('AgeInMonth == AgeInMonth.min()')[['AgeInMonth','AgeinYears','AgeTwoPlus']]

,AgeInMonth,AgeinYears,AgeTwoPlus
317,0,0,1
896,0,0,1
1019,0,0,1
1093,0,0,1
1126,0,0,1
...,...,...,...
42756,0,0,1
42996,0,0,1
43233,0,0,1
43247,0,0,1


It turns out that the largest age in our dataset is 17 years which is 215 months, while the smallest age is 0 age in month.

The average age in years is 6.5 while the median is 5.

In [65]:
tbi_nonmiss.groupby('AgeTwoPlus').agg({'PatNum':'count', 'AgeInMonth':['min','max','mean'], 'AgeinYears':['min','max','mean']})

PatNum AgeInMonth                  AgeinYears              
            count        min  max        mean        min max      mean
AgeTwoPlus                                                            
1           10901          0   23   11.588295          0   1  0.501055
2           32478         24  215  108.855687          2  17  8.620420

It becomes more clear that we have 25% age < 2 years old and 75% age >=2 years

Although these three columns describe the same identity, in the cleaning stage, I want to keep them all.

#### Findings

We have check the parent-child consistency between CT and findings.

Then I want to check whether the PosCT(TBI on CT (determined by PI)) is consistent with its definition (TBI on CT is defined as any of the traumatic findings (1-23) below, except for skull fracture.  Skull fractures were not regarded as TBIs unless the fracture was depressed by  at least the width of the skull.)

In [66]:
# case 1 : PosCT is yes (1) then at least one of Finding(1-23) should be yes (1) except finding 11 (skull fracture)
ct_findings = [col for col in tbi_nonmiss.columns if col.startswith('Finding') and col != 'Finding11']
ct_inconsistencies_case1 = tbi_nonmiss.query('PosCT == 1 and not '
'((' + ' | '.join([f'Finding{i} == 1' for i in [ int(i[7:]) for i in tbi_nonmiss.columns if i.startswith('Finding') and i != 'Finding11' ] ]) + '))')\
    [['PosCT'] + ct_findings]
ct_inconsistencies_case1.shape

(33, 18)

In [67]:
ct_inconsistencies_case1.describe(include='all')

,PosCT,Finding1,Finding2,Finding3,Finding4,Finding5,Finding6,Finding7,Finding8,Finding9,Finding10,Finding12,Finding13,Finding14,Finding20,Finding21,Finding22,Finding23
count,33,33,33,33,33,33,33,33,33,33,33,33,33,33,33,33,33,33
unique,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
top,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
freq,33,33,33,33,33,33,33,33,33,33,33,33,33,33,33,33,33,33


In [68]:
# case 2 : PosCT is no (0) then all of Finding(1-23) should be no (0)
ct_inconsistencies_case2 = tbi_nonmiss.query('PosCT == 0 and '
'((' + ' | '.join([f'Finding{i} == 1' for i in [ int(i[7:]) for i in tbi_nonmiss.columns if i.startswith('Finding') ] ]) + '))')\
    [['PosCT'] + ct_findings]
ct_inconsistencies_case2.shape

(499, 18)

In [69]:
ct_inconsistencies_case2.describe(include='all')

,PosCT,Finding1,Finding2,Finding3,Finding4,Finding5,Finding6,Finding7,Finding8,Finding9,Finding10,Finding12,Finding13,Finding14,Finding20,Finding21,Finding22,Finding23
count,499,499,499,499,499,499,499,499,499,499,499,499,499,499,499,499,499,499
unique,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
top,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
freq,499,499,499,499,499,499,499,499,499,499,499,499,499,499,499,499,499,499


It seems that the discrepancy happens in Finding 11 (skull structure). Going back to the definition of TBI on CT regarding skull structure, when the fracture was depressed by  at least the width of the skull, it will be considered TBI. Otherwise, it's not a TBI on CT. It really explains that when Finding 11 is yes while other findings are all no, some are categorized as TBI while others are not.

#### ciTBI Results

`PosIntFinal` should be in line with its definition: Clinically-important TBI was defined as having at least one of the following: (1) neurosurgical procedure performed, (2)  intubated > 24 hours for head trauma,  (3) death due to TBI or in the ED, (4) hospitalized for >= 2 nights due to head injury and having a TBI on CT.


In [70]:
# case 1: PosIntFinal is yes (1) then DeathTBI or HospHeadPosCT or Intub24Head or Neurosurgery should be yes (1)
int_final_inconsistencies_case1 = tbi_nonmiss.query('PosIntFinal == 1 and not (DeathTBI == 1 or HospHeadPosCT == 1 or Intub24Head == 1 or Neurosurgery == 1)')\
    [['PosIntFinal','DeathTBI','HospHeadPosCT','Intub24Head','Neurosurgery']]
int_final_inconsistencies_case1.shape

(3, 5)

In [71]:
int_final_inconsistencies_case1

,PosIntFinal,DeathTBI,HospHeadPosCT,Intub24Head,Neurosurgery
6667,1.0,0.0,0,0.0,0.0
16529,1.0,0.0,0,0.0,0.0
19454,1.0,0.0,0,0.0,0.0


There exist three cases who didn't have any of the 4 behaviors but were considered as has ciTBI. (Invalid)

In [72]:
# case 2 : PosIntFinal is no (0) then DeathTBI and HospHeadPosCT and Intub24Head and Neurosurgery should be no (0)
int_final_inconsistencies_case2 = tbi_nonmiss.query('PosIntFinal == 0 and (DeathTBI == 1 or HospHeadPosCT == 1 or Intub24Head == 1 or Neurosurgery == 1)')\
    [['PosIntFinal','DeathTBI','HospHeadPosCT','Intub24Head','Neurosurgery']]
int_final_inconsistencies_case2.shape

(0, 5)

In [73]:
def flag_posintfinal_inconsistency(
    df: pd.DataFrame,
    outcome_col: str = "PosIntFinal",
    symptom_cols = [
        "DeathTBI",
        "HospHeadPosCT",
        "Intub24Head",
        "Neurosurgery"
    ],
    flag_col: str = "PosIntFinal_invalid"
) -> pd.DataFrame:
    """
    Flag logically inconsistent cases where:
    PosIntFinal == 1 but all symptom indicators are 0.

    Does NOT change PosIntFinal — only adds a flag column.

    Returns a modified copy.
    """
    out = df.copy()

    # Define "no symptoms present"
    no_symptoms = (out[symptom_cols] == 0).all(axis=1)

    # Invalid if outcome=1 but no symptoms
    out[flag_col] = (
        (out[outcome_col] == 1) & no_symptoms
    ).astype('int')  # 'c' for category (or use int if you prefer)

    return out

In [74]:
tbi_flagged = flag_posintfinal_inconsistency(tbi_nonmiss)
tbi_flagged[['PosIntFinal','DeathTBI','HospHeadPosCT','Intub24Head','Neurosurgery','PosIntFinal_invalid']]\
    .query('PosIntFinal_invalid == 1')

,PosIntFinal,DeathTBI,HospHeadPosCT,Intub24Head,Neurosurgery,PosIntFinal_invalid
6667,1.0,0.0,0,0.0,0.0,1
16529,1.0,0.0,0,0.0,0.0,1
19454,1.0,0.0,0,0.0,0.0,1


We identified a small number of logically inconsistent records in which PosIntFinal = 1 despite none of the four defining clinical criteria being present. Rather than altering the outcome label, we retained PosIntFinal as recorded and created a binary indicator (PosIntFinal_invalid) to flag these cases. This preserves the original labeling while allowing us to assess the impact of these inconsistencies in downstream analyses.

## Step 4: Clean and pre-process the data

In this part, I will write a **clean** function and a **preprocesss** function to carry out all the strategies I have explore above: 

> **clean_data**: it will include all the data integrity steps I will never want to change between EDA and Modeling
> **preprocess_data**: it will include all the options that will differ based on the EDA or model requiremnt


***Clean***


+ **Convert Data Type**:
    Convert the data types to comply with the documents. There are just two types: numerical and categorical.
    + binary category: `int`
    + multiple category: `category`


+ **Drop samples with missing final result**: 
    Our goal is to make clinical decision rules which can identify the patients with high risk of clinically-important traumatic brain injuries, so samples without the final result is meaningless from certain perspective. Also, there are just 18 patients with missing parimary outcome, which is relatively very small in our total samples. 

+ **Flag final outcome inconsistentcy**: 
  There exist three cases who didn't have any of the 4 behaviors but were considered as has ciTBI. I choose to add a flag column to represent the inconsistency.

***Preprocess***

+ **Impute GCS scores**: 
    For GCS scores missingness, I deal with it with 3 different situations. 
    + Case 1: all components missing but total == 15 -> assume max components
    + Case 2: total known, exactly one component missing -> infer
    + Case 3: total known but only 1 or 0 components known -> ambiguous. In this case, the information is insufficient to impute reliably, so I retain missingness rather than making an arbitrary guess. The alternative judgment call is fill with the mode(median/mean) which has the risk of bringing about inconsistency between total and its components.

+ **Impute parent-child pairs**: 
    For parent-child pair variables, I apply the general strategy consistent with orginial data collection: 
  + Case 1: parent missing 
      + children = missing_code `92`; 
      + add an missingness indicator column;
      + fill with 0/leave it alone
  
  + Case 2: parent == yes and child missing
    + leave/add `missing` category

+ **Impute categorical varibales**: Since missingness sometimes is also informative, there are two ways:
  + Binary: add an missingness indicator, fill with 0/ leave it alone
  + Multiple: add a missing category/ leave it alone
  

+ **Exclude patients with GCS 3-13**: 
  it's a judgment call. In principle, we shouldn't exclude patients with meaningful variables in cleaning stage. But since in Kuppermann's way, they exclude patients with GCS 3-13 before derviartion, I just include this option for future modeling use.



#### Helper Function

we will impelement all the helper functions we have defined before

+ `impute_gcs`
+ `fit_imputer`, `apply_imputer`
+ `apply_parent_child_missingness`
+ `handle_missing_for_categories`

#### Function: Clean data

In [75]:

def clean_data(
        df: pd.DataFrame,
        numeric_cols: List[str],
        binary_cols: List[str],
        multi_category_cols: List[str],
        target_col: str = 'PosIntFinal',
        drop_target_na: bool = True
):
    # Step 1: coerce data types
    out = df.copy()
    out[numeric_cols] = out[numeric_cols].apply(pd.to_numeric, errors='coerce')
    out[binary_cols] = out[binary_cols].astype('int', errors='ignore')  # if already numeric, keep as is; if not, try to convert to int (0/1), coercing errors to NaN
    out[multi_category_cols] = out[multi_category_cols].astype('category')


    # Step 2: Drop rows with missing target
    if drop_target_na:
        out = out.dropna(subset=[target_col])


    # Step 3: Flag inconsistencies in PosIntFinal
    out = flag_posintfinal_inconsistency(out)
    return out


##### Test 1: clean function

In [76]:
# test the function
# by default, we will drop rows with missing target for our analysis
# for all the missing values, I will add a indicator column or add a missing category, and not fill then with any values
tbi_cleaned = clean_data(
    df=tbi_origin,
    numeric_cols=numeric_cols,
    binary_cols=binary_cols,
    multi_category_cols=multi_category_cols,
    target_col='PosIntFinal',
    drop_target_na=True
)

quick check about the cleaned dataset

In [77]:
tbi_cleaned.shape

(43379, 126)

In [78]:
tbi_cleaned['PosIntFinal'].isnull().sum()

np.int64(0)

We have removed the null final result.

In [79]:
tbi_cleaned.dtypes

PatNum                    int64
EmplType                float64
Certification             int64
InjuryMech             category
High_impact_InjSev     category
                         ...   
HospHeadPosCT             int64
Intub24Head             float64
Neurosurgery            float64
PosIntFinal             float64
PosIntFinal_invalid       int64
Length: 126, dtype: object

In [80]:
tbi_cleaned.select_dtypes(include='number').nunique()

PatNum                 43379
EmplType                   5
Certification              5
Seiz                       2
ActNorm                    2
Vomit                      2
Dizzy                      2
Intubated                  2
Paralyzed                  2
Sedated                    2
GCSEye                     4
GCSVerbal                  5
GCSMotor                   6
GCSTotal                  13
GCSGroup                   2
AMS                        2
FontBulg                   2
SFxBas                     2
Hema                       2
Clav                       2
NeuroD                     2
OSI                        2
Drugs                      2
CTForm1                    2
AgeInMonth               216
AgeinYears                18
AgeTwoPlus                 2
Gender                     2
Ethnicity                  2
Observed                   2
CTDone                     2
DeathTBI                   2
HospHead                   2
HospHeadPosCT              2
Intub24Head   

#### Function: Preprocess_data

In [81]:
from typing import Dict, List, Optional, Literal
from sklearn.model_selection import train_test_split

GCSFillingStrategy = Literal["mode", "median", "mean",'leave'] 
ParentMissingFill = Literal["leave", "fill0"]
ChildMissingStrategy = Literal["leave", "missing_category"]
BinaryMissing = Literal["leave", "fill0"]
MultiMissingStrategy = Literal["leave", "missing_category"]


def preprocess_data(
    df: pd.DataFrame,
    numeric_cols,
    categorical_cols,
    multi_category_cols: List[str],
    binary_cols: List[str],
    parent_child_groups: Dict[str, List[str]],
    target_col: str = "PosIntFinal",
    test_size: float = 0.2,
    val_size: float = 0.1,              # fraction of FULL data (not of train_temp)
    random_state: int = 42,
    stratify_col: List[str] = ["PosIntFinal", "AgeTwoPlus"],

    if_exclude_gcs_under_13: bool = True,  # if True, drop rows with gcs 3-13/GCSGroup =1 

    if_fill_missing_gcs: bool = True,      # if True, infer missing GCS components using a consistent strategy learned from TRAIN only (to avoid leakage)
    gcs_fill_strategy: GCSFillingStrategy = "mode",
    if_one_hot_encode: bool = False,
    gcs_cols: Optional[List[str]] = ["GCSEye","GCSVerbal","GCSMotor"],
    drop_first_cat_in_ohe: bool = True,

    if_drop_na_rows: bool = False,          # if True, drop rows with any missing values in numeric_cols or categorical_col

    if_handle_parent_child_missing: bool = True,  # if True, apply the parent-child missingness handling for specified groups
    if_handle_missing_for_categories: bool = True,  # if True, apply the standalone categorical missingness handling for specified columns
    missing_category_label: str = "missing",
    if_add_flag_missing_cols: bool = True,
    parent_missing_strategy: ParentMissingFill = "leave",
    child_missing_when_parent_yes: ChildMissingStrategy = "missing_category",
    binary_missing_strategy: BinaryMissing = "leave",
    multi_missing_strategy: MultiMissingStrategy = "missing_category",
) -> Dict[str, object]:
    
    out = df.copy()

    # optional: exclude rows with GCS under 13 (GCSGroup=1) if specified
    if if_exclude_gcs_under_13:
        out = out[out['GCSGroup'] != 1] 

    # optional: handle missing values in categorical variables
    if handle_missing_for_categories:
        out = handle_missing_for_categories(
            out,
            binary_cat_cols= binary_cols,
            multi_cat_cols= multi_category_cols,
            binary_missing_strategy=binary_missing_strategy,
            add_binary_missing_indicator=if_add_flag_missing_cols,
            missing_category_label=missing_category_label,
            multi_missing_strategy=multi_missing_strategy,
            force_to_category=True
        )


    # optional: handle parent-child missingness for specified groups
    if if_handle_parent_child_missing:
        out = apply_parent_child_missingness(out, parent_child_groups, missing_code=92, parent_yes_value=1, 
                                                  add_parent_missing_indicator=if_add_flag_missing_cols, 
                                                  parent_missing_strategy=parent_missing_strategy, 
                                                  child_missing_when_parent_yes=child_missing_when_parent_yes, 
                                                  missing_category_label=missing_category_label)

    # for missingness flag, we do it before train-test split to capture the missingness pattern in the whole dataset to make sure we have the same features across train/val/test. 
    
    
    #  train/val/test split (stratify if po)
    # do this BEFORE any imputation/encoding to avoid leakage

    train_temp, test_df = train_test_split(
        out,
        test_size=test_size,
        random_state=random_state,
        stratify=out[stratify_col] if stratify_col is not None else None,
    )

    # val_size is fraction of full dataset; convert to fraction of train_temp
    val_frac_of_train_temp = val_size / (1.0 - test_size)

    train_df, val_df = train_test_split(
        train_temp,
        test_size=val_frac_of_train_temp,
        random_state=random_state,
        stratify=train_temp[stratify_col] if stratify_col is not None else None,
    )

    # optional: exclude rows with GCS under 13 (GCSGroup=1) if specified
    if if_exclude_gcs_under_13:
        train_df = train_df[train_df['GCSGroup'] != 1]
        val_df   = val_df[val_df['GCSGroup'] != 1]
        test_df  = test_df[test_df['GCSGroup'] != 1]


    # optional: infer GCS components and fill missing GCS values using a consistent strategy learned from TRAIN only (to avoid leakage)
    if if_fill_missing_gcs and not gcs_cols:
        raise ValueError("if_fill_missing_gcs=True but gcs_cols is None/empty")
    if if_fill_missing_gcs:
        # first impute GCS components using the impute_gcs function we defined earlier, which uses the internal consistency rules of GCS to fill in some missing values
        train_df = impute_gcs(train_df)
        val_df   = impute_gcs(val_df)
        test_df  = impute_gcs(test_df)

        if gcs_fill_strategy != 'leave':
            # then, for any remaining missing GCS component values, fill them using a simple strategy (mode/median/mean) learned from TRAIN only
            imputer_values = fit_imputer(train_df, gcs_cols, filling_strategy=gcs_fill_strategy)
            train_df = apply_imputer(train_df, imputer_values)
            val_df   = apply_imputer(val_df, imputer_values)
            test_df  = apply_imputer(test_df, imputer_values)
        else:
            imputer_values = None

    # optional: one-hot encode multi-category variables
    
    if if_one_hot_encode:
        # pick which columns to one-hot (usually multi-cat)
        ohe_cols = [c for c in (multi_category_cols) if c in train_df.columns]

        train_ohe = pd.get_dummies(train_df, columns=ohe_cols, drop_first=drop_first_cat_in_ohe)
        val_ohe   = pd.get_dummies(val_df, columns=ohe_cols, drop_first=drop_first_cat_in_ohe)
        test_ohe  = pd.get_dummies(test_df, columns=ohe_cols, drop_first=drop_first_cat_in_ohe)

        # align val/test to train columns
        val_ohe  = val_ohe.reindex(columns=train_ohe.columns, fill_value=0)
        test_ohe = test_ohe.reindex(columns=train_ohe.columns, fill_value=0)

        train_df, val_df, test_df = train_ohe, val_ohe, test_ohe
    
    # optional: drop rows with any missing values in numeric_cols or categorical_col
    if if_drop_na_rows:
        train_df = train_df.dropna(subset=numeric_cols+categorical_cols)
        val_df   = val_df.dropna(subset=numeric_cols+categorical_cols)
        test_df  = test_df.dropna(subset=numeric_cols+categorical_cols)

    return train_df, val_df, test_df


##### Test 2: Preprocessed Data

In [82]:
# check the prepare_data function
train_processed, val_processed, test_processed = preprocess_data(
    df=tbi_cleaned,
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols,
    multi_category_cols=multi_category_cols,    
    binary_cols=binary_cols,
    parent_child_groups=parent_child_groups,
    target_col='PosIntFinal',
    test_size=0.2,
    val_size=0.1,
    random_state=42,
    stratify_col=['PosIntFinal', 'AgeTwoPlus'],
    if_exclude_gcs_under_13=True,
    if_fill_missing_gcs=True,
    gcs_fill_strategy="mode",
    if_one_hot_encode=False,
    drop_first_cat_in_ohe=True,
    if_drop_na_rows=False,
    if_handle_parent_child_missing=True,
    if_handle_missing_for_categories=True,
    missing_category_label="missing",
    if_add_flag_missing_cols=True,
    parent_missing_strategy="fill0",
    child_missing_when_parent_yes="missing_category",
    binary_missing_strategy="fill0",
    multi_missing_strategy="missing_category"
)

In [83]:
print("Train shape:", train_processed.shape)
print("Val shape:", val_processed.shape)
print("Test shape:", test_processed.shape)

Train shape: (29687, 148)
Val shape: (4242, 148)
Test shape: (8483, 148)


In [84]:
train_processed.isnull().sum().sort_values(ascending=False).head(10)

EmplType         15
Finding1          0
Ethnicity         0
Race              0
Observed          0
EDDisposition     0
CTDone            0
EDCT              0
PosCT             0
PatNum            0
dtype: int64

In [85]:
val_processed.isnull().sum().sort_values(ascending=False).head(10)

PatNum           0
Gender           0
Race             0
Observed         0
EDDisposition    0
CTDone           0
EDCT             0
PosCT            0
Finding1         0
Finding2         0
dtype: int64

In [86]:
test_processed.isnull().sum().sort_values(ascending=False).head(10)

EmplType         3
Finding1         0
Ethnicity        0
Race             0
Observed         0
EDDisposition    0
CTDone           0
EDCT             0
PosCT            0
PatNum           0
dtype: int64

During cleaning and preprocessing, I don't deal with the three identifier columns: 'PatNum','EmplType','Certification'. 

So, the missingness in those columns are reasonable.

### Test the clean and prerocessing pipline

I have compiled the `clean_data` and `preprocess_data` function separately in `clean.py` and `preprocess.py`. Meanwhile, I have prepared a `features.py` which includes all the column categories like numerical, binary and parent_child_group as list. Thus, whenever I want to use them in my following procedure, I can directly import them.

In [87]:
from clean import clean_data
from preprocess import preprocess_data
from features import numeric_cols, binary_cols, multi_category_cols, parent_child_groups, categorical_cols
# Step 1: Clean the raw data
tbi_cleaned_new = clean_data(
    df=tbi_origin,
    numeric_cols=numeric_cols,
    binary_cols=binary_cols,
    multi_category_cols=multi_category_cols,
    target_col='PosIntFinal',
    drop_target_na=True
)

In [88]:
tbi_cleaned_new.shape

(43379, 126)

In [89]:
tbi_origin[tbi_origin['PosIntFinal'].isna()].shape

(20, 125)

In the cleaning stage, we just drop 20 rows which has missing target columns.

In [90]:
# for the test, I exclude GCS under 13 and fill missing GCS values using the imputation strategy, and handle parent-child missingness, but I will not do one-hot encoding or drop rows with missing values to see how the function works with all the options.
train_preprocessed_new, val_preprocessed_new, test_preprocessed_new = preprocess_data(
    df=tbi_cleaned_new,
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols,
    multi_category_cols=multi_category_cols,
    binary_cols=binary_cols,
    parent_child_groups=parent_child_groups,
    target_col='PosIntFinal',
    test_size=0.2,
    val_size=0.1,
    random_state=42,
    stratify_col=['PosIntFinal', 'AgeTwoPlus'],
    if_exclude_gcs_under_13=True,
    if_fill_missing_gcs=True,
    gcs_fill_strategy="mode",
    if_one_hot_encode=False,
    drop_first_cat_in_ohe=True,
    if_drop_na_rows=False,
    if_handle_parent_child_missing=True,
    if_handle_missing_for_categories=True,
    missing_category_label="missing",
    if_add_flag_missing_cols=True,
    parent_missing_strategy="fill0",
    child_missing_when_parent_yes="missing_category",
    binary_missing_strategy="fill0",
    multi_missing_strategy="missing_category"
)

In [91]:
train_preprocessed_new.shape, val_preprocessed_new.shape, test_preprocessed_new.shape

((29687, 148), (4242, 148), (8483, 148))

In [92]:
# sanity check: make sure the overall numder of rows in train/val/test adds up to the total in the paper 42412
total_rows = train_preprocessed_new.shape[0] + val_preprocessed_new.shape[0] + test_preprocessed_new.shape[0]
print(f"Total rows in preprocessed data: {total_rows}")
print(f"Original cleaned data rows: {tbi_cleaned_new.shape[0]}")

Total rows in preprocessed data: 42412
Original cleaned data rows: 43379


In [93]:
train_preprocessed_new.isnull().sum().sort_values(ascending=False).head(10)

EmplType         15
Finding1          0
Ethnicity         0
Race              0
Observed          0
EDDisposition     0
CTDone            0
EDCT              0
PosCT             0
PatNum            0
dtype: int64

In [94]:
val_preprocessed_new.isnull().sum().sort_values(ascending=False).head(10)

PatNum           0
Gender           0
Race             0
Observed         0
EDDisposition    0
CTDone           0
EDCT             0
PosCT            0
Finding1         0
Finding2         0
dtype: int64

In [95]:
test_preprocessed_new.isnull().sum().sort_values(ascending=False).head(10)

EmplType         3
Finding1         0
Ethnicity        0
Race             0
Observed         0
EDDisposition    0
CTDone           0
EDCT             0
PosCT            0
PatNum           0
dtype: int64

For the missingness after the preprocess, we only have missing values in identifier columns which is what I expect.